# Setting

In [1]:
from src.utils.settings import settings

vectorstore_setting = settings.vector_store
embedding_setting = settings.embedding
qdrant_setting = settings.vector_store

# Embedding

In [2]:
from src.core.data.embeddings import EmbeddingFactory

model_name = embedding_setting.model_name
dimensions = embedding_setting.dimensions

embedding_config = {
    "model_name" : model_name,
    "dimensions" : dimensions
}

embedding = EmbeddingFactory.get(name = "AITeamVN/Vietnamese_Embedding", **embedding_config)

/Users/hung/Desktop/scanx-agent/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6705.86it/s]


# Qdrant client

In [3]:
from src.core.data.documents.factory import DocumentStoreFactory

collection_name = qdrant_setting.collection_name
persist_directory = qdrant_setting.persist_directory
vector_size = qdrant_setting.vector_size
distance = qdrant_setting.distance

qdrant_config = {
    "collection_name" : collection_name, 
    "qdrant_config" : {"path":persist_directory},
    "vector_size" : vector_size,
    "distance" : distance
}

store = DocumentStoreFactory.get(
    name="qdrant",
    **qdrant_config
)

# Scorer

In [4]:
from src.core.data.scoring import ProductScorer
scorer = ProductScorer(threshold=60)

# Create Vector DB

In [5]:
# from src.core.data.ingestion import FnBDataIngestor

# # Init ingestor (inject all dependencies)
# ingestor = FnBDataIngestor(
#         document_store=store,
#         embedding=embedding,
#         scorer=scorer,
#     )

# ingestor.ingest_from_file(file_path="external_data_storage/fnb.json", collection_name=collection_name)

# Create Agent

In [8]:
# FnB Agent
## model - brain
from src.core.llm import LLMChatModel
model = LLMChatModel.get(provider = "openai", temperature = 0.0)

## prompts
from src.core.prompts.fnb_prompt import FNB_SYSTEM_PROMPT
system_prompt = FNB_SYSTEM_PROMPT

## tools
from src.agents.fnb.utils.tools import create_menu_search_tool, create_get_recommendations_tool, create_get_product_detail_tool
menu_search_tool = create_menu_search_tool(
    store,
    embedding,
    collection_name
)
get_product_detail_tool = create_get_product_detail_tool(
    "external_data_storage/fnb.json"
)
get_recommendations_tool = create_get_recommendations_tool(
    "external_data_storage/fnb.json"
)
tools = [menu_search_tool, get_product_detail_tool, get_recommendations_tool]

# agent initialization
from src.agents.fnb.agent import ReActAgent
fnb_agent = ReActAgent(
    model = model,
    tools = tools,
    system_prompt = system_prompt,
)

# INFERENCE

In [9]:
async def run(query, configurable):
    print("USER:",query)
    print("BOTBUDDY:")
    async for chunk in fnb_agent.astream(message = query, configurable = configurable):
        mode = chunk["mode"]
        node = chunk["node"]
        if mode == "messages":
            print(chunk["content"], end="", flush=True)

        elif mode == "updates":
            if "model" in chunk["state"]:
                print("\n" + "..........TOOL CALLING..........")
                print(chunk["state"]["model"]["messages"][-1].tool_calls)
                print("."*50)
            else:
                print("\n" + "..........TOOL RESULTS..........")
                print(chunk["state"]["tools"]["messages"][-1].content)
                print("."*50)

In [10]:
import random
configurable = {
    "tenant_id": "tenant-1",
    "thread_id": "thread-" + str(random.randint(1000,9999)),
}

In [11]:
q1 = "Bây giờ tôi chưa biết nên ăn gì, chỉ biết muốn ăn gì đó ngọt một chút, hãy tìm cho tôi món như vậy và gợi ý thử 2 món ăn kèm"

await run(q1, configurable)

USER: Bây giờ tôi chưa biết nên ăn gì, chỉ biết muốn ăn gì đó ngọt một chút, hãy tìm cho tôi món như vậy và gợi ý thử 2 món ăn kèm
BOTBUDDY:

..........TOOL CALLING..........
[{'name': 'menu_search', 'args': {'query': 'ngọt', 'top_k': 5}, 'id': 'call_kdllxpjm', 'type': 'tool_call'}]
..................................................

..........TOOL RESULTS..........
--- Kết quả 1 (Score: 0.3416) ---
ID: item-016
Tên: Canh Kimchi đậu hũ non
Giá: 110,000đ
Danh mục: Món ăn kèm
Mô tả: Tên món: Canh Kimchi đậu hũ non
Danh mục: Món ăn kèm
Mô tả: Nước dùng chua cay nóng hổi, đậu hũ mềm mượt tan trong miệng. Rất hợp để húp xì xụp sau khi ăn thịt.
Nguyên liệu: Kimchi, Đậu hũ non, Ba chỉ heo
Đặc điểm: Cay, Nóng
Phục vụ: Trưa/Tối
Giá: 110,000đ
Tags: Cay, Nóng
Thời gian phục vụ: Trưa/Tối

--- Kết quả 2 (Score: 0.3064) ---
ID: item-025
Tên: Nước suối khoáng
Giá: 20,000đ
Danh mục: Đồ uống
Mô tả: Tên món: Nước suối khoáng
Danh mục: Đồ uống
Mô tả: Nước khoáng tinh khiết đóng chai lạnh, làm sạch vị giá

In [ ]:
# q1 = "tôi muốn gì đó mặn mặn"
# q2 = "Nguyên liệu của món đầu tiên trên đó gồm những gì?"
# q3 = "tôi sẽ chọn món này"

# await run(q1, configurable)
# print("="*50)
# await run(q2, configurable)
# print("="*50)
# await run(q3, configurable)

USER: tôi muốn gì đó mặn mặn
BOTBUDDY:

..........TOOL CALLING..........
[{'name': 'menu_search', 'args': {'query': 'mặn', 'top_k': 5}, 'id': 'call_7xkehp4r', 'type': 'tool_call'}]
..................................................

..........TOOL RESULTS..........
--- Kết quả 1 (Score: 0.3078) ---
ID: item-008
Tên: Mực lá nướng sa tế
Giá: 210,000đ
Danh mục: Hải sản nướng
Mô tả: Tên món: Mực lá nướng sa tế
Danh mục: Hải sản nướng
Mô tả: Mực tươi dày mình, khi nướng lên có độ dai giòn, vị ngọt của biển cả hòa quyện cùng sa tế cay tê.
Nguyên liệu: Mực lá tươi, Sa tế tôm, Sả
Đặc điểm: Cay, Hải sản
Phục vụ: Cả ngày
Giá: 210,000đ
Tags: Cay, Hải sản
Thời gian phục vụ: Cả ngày

--- Kết quả 2 (Score: 0.2979) ---
ID: item-013
Tên: Nấm tổng hợp nướng giấy bạc
Giá: 85,000đ
Danh mục: Món ăn kèm
Mô tả: Tên món: Nấm tổng hợp nướng giấy bạc
Danh mục: Món ăn kèm
Mô tả: Nấm kim châm và nấm đùi gà được nướng trong giấy bạc giữ trọn nước ngọt, thơm mùi bơ tỏi.
Nguyên liệu: Nấm kim châm, Nấm đùi gà, Bơ, Tỏi
Đặc điểm: Chay, Ngọt thanh
Phục vụ: Cả ngày
Giá: 85,000đ
Tags: Chay, Ngọt thanh
Thời gian phục vụ: Cả ngày

--- Kết quả 3 (Score: 0.2878) ---
ID: item-009
Tên: Tôm sú nướng muối ớt
Giá: 240,000đ
Danh mục: Hải sản nướng
Mô tả: Tên món: Tôm sú nướng muối ớt
Danh mục: Hải sản nướng
Mô tả: Tôm chắc thịt, vỏ ngoài giòn tan thấm vị muối ớt cay nồng. Rất phù hợp làm món nhắm cùng bia.
Nguyên liệu: Tôm sú, Muối hột, Ớt tươi
Đặc điểm: Cay, Hải sản
Phục vụ: Cả ngày
Giá: 240,000đ
Tags: Cay, Hải sản
Thời gian phục vụ: Cả ngày

--- Kết quả 4 (Score: 0.2723) ---
ID: item-017
Tên: Mì lạnh Naengmyeon
Giá: 135,000đ
Danh mục: Món chính
Mô tả: Tên món: Mì lạnh Naengmyeon
Danh mục: Món chính
Mô tả: Sợi mì dai giòn trong nước dùng bò lạnh có đá viên. Cực kỳ sảng khoái, làm dịu cái nóng của bếp nướng.
Nguyên liệu: Sợi mì kiều mạch, Nước dùng bò, Lê, Trứng luộc
Đặc điểm: Lạnh, Sảng khoái
Phục vụ: Trưa/Tối
Giá: 135,000đ
Tags: Lạnh, Sảng khoái
Thời gian phục vụ: Trưa/Tối

--- Kết quả 5 (Score: 0.2696) ---
ID: item-021
Tên: Bia Sapporo tươi
Giá: 55,000đ
Danh mục: Đồ uống
Mô tả: Tên món: Bia Sapporo tươi
Danh mục: Đồ uống
Mô tả: Vị bia êm đằm, lớp bọt mịn màng, cực kỳ sảng khoái khi uống kèm thịt nướng đậm đà.
Nguyên liệu: Mạch nha, Hoa bia
Đặc điểm: Lạnh, Có cồn
Phục vụ: Cả ngày
Giá: 55,000đ
Tags: Lạnh, Có cồn
Thời gian phục vụ: Cả ngày
..................................................
Chào bạn! 🌟

Bạn muốn “mặn mặn” – mình có 2 món “đậm vị biển” rất hợp khẩu vị:

| Món | Giá | Đặc điểm mặn | Ghi chú |
|-----|------|--------------|----------|
| **Mực lá nướng sa tế** (item-008) | 210.000 đ | Mực tươi dày mình, nướng vừa giòn vừa đậm vị biển, kết hợp với hương sa tế cay nhẹ – vị mặn ngọt hòa quyện. | Thích hợp nếu bạn muốn một chút “kick” nhẹ. |
| **Tôm sú nướng muối ớt** (item-009) | 240.000 đ | Tôm chắc thịt, lớp vỏ giòn tan thấm muối ớt – vị mặn đậm đặc cùng chút cay nồng. | Phù hợp khi muốn món “đậm vị” nhất, thường ăn kèm bia. |

🧃 **Gợi ý đồ uống kèm**  
- **Bia Sapporo tươi** (55.000 đ) – vị bia êm, bọt mịn, giúp cân bằng độ mặn và nhẹ nhàng “làm dịu” vị cay nếu bạn chọn món có sa tế hoặc muối ớt.  

Bạn muốn đặt ngay một trong hai món trên, hay muốn biết thêm chi tiết (nguyên liệu, cách ăn kèm…) cho món nào? Nếu có sở thích riêng (ít cay hơn, muốn ăn kèm salad,…) mình sẽ gợi ý thêm cho phù hợp. 😊
..........TOOL CALLING..........
[]
..................................................
==================================================
USER: Nguyên liệu của món đầu tiên trên đó gồm những gì?
BOTBUDDY:
**Món: Mực lá nướng sa tế** (item‑008)  

- **Nguyên liệu:**  
  1. Mực lá tươi  
  2. Sa tế tôm  
  3. Sả  

Món này mang vị biển mặn mà, kết hợp với hương cay nhẹ của sa tế và mùi thơm đặc trưng của sả. Bạn muốn biết thêm về cách ăn kèm hoặc muốn đặt món ngay không? 😊
..........TOOL CALLING..........
[]
..................................................
==================================================
USER: tôi sẽ chọn món này
BOTBUDDY:

..........TOOL CALLING..........
[{'name': 'get_recommendations', 'args': {'product_ids': ['item-008']}, 'id': 'call_mxlo640d', 'type': 'tool_call'}]
..................................................

..........TOOL RESULTS..........
=== Gợi ý kết hợp cho 1 sản phẩm đã chọn ===


• Bia Sapporo tươi — 55,000đ
  Danh mục: Đồ uống
  Mô tả: Vị bia êm đằm, lớp bọt mịn màng, cực kỳ sảng khoái khi uống kèm thịt nướng đậm đà.
  Tags: Lạnh, Có cồn
  Lý do gợi ý: Kết hợp tốt với: Mực lá nướng sa tế

• Lẩu Thái hải sản — 450,000đ
  Danh mục: Lẩu
  Mô tả: Vị chua cay mặn ngọt bùng nổ, đầy ắp hải sản tươi sống. Thích hợp cho nhóm 2-3 người ăn kết thúc bữa.
  Tags: Cay, Nhóm đông
  Lý do gợi ý: Kết hợp tốt với: Mực lá nướng sa tế


Tổng cộng 2 gợi ý kết hợp.
..................................................
Rất tuyệt! 🎉  

**Bạn vừa chọn:**  
**Mực lá nướng sa tế** – 210.000 đ  
(nguyên liệu: mực lá tươi, sa tế tôm, sả)

**Gợi ý kèm theo** để bữa ăn thêm trọn vẹn:  

| Đồ uống / Món phụ | Giá | Lý do đề xuất |
|-------------------|------|----------------|
| **Bia Sapporo tươi** | 55.000 đ | Bia êm đằm, bọt mịn, cân bằng vị mặn và hơi cay của sa tế. |
| **Lẩu Thái hải sản** | 450.000 đ | Lẩu chua‑cay‑mặn ngọt, đầy hải sản tươi, thích hợp nếu muốn chia sẻ sau khi nướng. |

Bạn muốn **thêm Bia Sapporo**, **lẩu Thái** hay chỉ giữ nguyên **Mực lá nướng sa tế**?  
Hãy cho mình biết để mình chuẩn bị đặt món cho bạn ngay nhé. 😊
..........TOOL CALLING..........
[]
..................................................
